# Sana — LoRA experiments on Colab (thin shell over the repo runner)

Trains LoRAs for **[Sana](https://github.com/NVlabs/Sana)** (NVlabs; Xie et al. 2024, arXiv 2410.10629) through
model type `sana` in the [AbstractEyes diffusion-pipe fork](https://github.com/AbstractEyes/diffusion-pipe), loads
every saved LoRA back through diffusers, measures what it does, and uploads each experiment into its own folder of
**[AbstractPhil/geolip-beatrix-sana](https://huggingface.co/AbstractPhil/geolip-beatrix-sana)** as it goes.

**No dataset needed.** The stock Sana 600M 512px model renders its own training sets: images of one mood captioned
with the *neutral* prompt ("a photo of …"). A LoRA can only lower its loss by giving neutral prompts that mood, so a
working pipeline shows up as a mood shift on prompts that never mention one. A quarter of the scenes never enter
training; every LoRA is scored on them with a CLIP mood judge (similarity to upbeat phrases minus similarity to
downbeat phrases), paired cell by cell against the same prompt and seed without the LoRA.

**`s.run_sequence()`** runs these one after another (each the first recipe with one change):

| folder | training images | what it asks |
|---|---|---|
| `e004_lora_mood_up` | upbeat | does it work? |
| `e005_lora_neutral_control` | the model's own neutral images | does fine-tuning alone move the mood? (a control that can fail) |
| `e006_lora_mood_down` | downbeat | does it go the other way? |
| `e007_lora_mood_up_reseed` | a fresh draw of upbeat images | does it replicate? |
| `e008_lora_mood_up_lr_5e-5` | upbeat, half the learning rate | how does it depend on the rate? |
| `e009_lora_mood_up_lr_2e-4` | upbeat, double the learning rate | (same) |
| `e010` … `e013` (`_draw2`) | the control, the mirror and the two learning rates on e007's draw of images | does each read hold on a second draw? |

Each arm's verdict: **FLAVOR LORA** (the mean moves the expected way, at least 75% of the 32 cells that way, beyond
3 standard errors), **NO EFFECT** (within 2 standard errors of 0, or under 60% that way), or **MIXED**. For scale: on
the stock model, writing upbeat words into the prompt moves the same score +1.81 over the neutral prompt.

**Same thin-shell contract as the Anima notebooks:** all logic is in the repo
(`geolip_anima_trainer.sana_runner.SanaRunner`); this notebook is clone → install → a few `s.<step>()` calls.
**To change anything, edit the repo and `git pull`; the notebook stays as it is.**

**Before you start:** pick a GPU runtime (the RTX PRO 6000 is the target) and put a **write** `HF_TOKEN` in Colab
Secrets (🔑) with notebook access: the uploads need it, and the sequence stops at once if it cannot write.

> **Licences.** Sana's weights are Apache-2.0; its Gemma-2-2B-IT text encoder is under Google's
> [Gemma Terms of Use](https://ai.google.dev/gemma/terms).

## 0 · Confirm the GPU + disk

The sequence needs about 30 GB of disk (the ~8.5 GB checkpoint, the dependencies, four training sets, the LoRAs).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!df -h /content | tail -1

## 1 · Bootstrap — clone + install (→ restart ONCE the first time)

Clones this repo and the diffusion-pipe fork (at `external/diffusion-pipe-fork`) and installs the dependencies (the
logic is in `anima_colab.install`, so a `git pull` updates it with no notebook edit). **Run the cell; if the kernel
restarts, run it again** (it then skips), then continue.

In [ ]:
import os, sys, subprocess
REPO = os.environ.get("ANIMA_REPO", "/content/anima-trainer")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/AbstractEyes/anima-trainer.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)   # the latest runner before importing it
sys.path.insert(0, REPO)
import anima_colab
if anima_colab.install(REPO, dp_url=anima_colab.DP_FORK_URL, similarity=False):   # + the fork that has Sana
    os.kill(os.getpid(), 9)             # -> restart; reconnect and RE-RUN this cell (then it skips)

## 2 · (after restart) Run the sequence

`setup()` checks the GPU, finds the fork and downloads the model. `run_sequence()` renders the four training sets and
the no-LoRA baseline first, then trains and evaluates each arm in order; every arm uploads its config, data list,
each saved epoch (as it is saved), previews, evaluation and log into `experiments/<folder>/` of the repo, and the
repo's README index updates after each arm. **A rerun skips every arm the repo already lists as done**, so after a
disconnect just run these two cells again. Run a subset with `s.run_sequence(["e004_lora_mood_up", ...])`.

In [ ]:
from geolip_anima_trainer.sana_runner import SanaRunner
s = SanaRunner()       # Sana 600M 512px; uploads to AbstractPhil/geolip-beatrix-sana (repo_id= to change)
s.setup()              # env + HF login + GPU check + the fork + the model download (~8.5 GB)

In [ ]:
s.run_sequence()       # e004 .. e009 in order; the trainer's log and each arm's reads print here

## Notes

- **Where things land:** the repo, one folder per experiment (README with the question, the recipe, the rule fixed
  before the run and the result; `meta.json`; `config/`; `data/`; `lora/epochN/`; `samples/`; `eval/`; `logs/`).
  Locally: `/content/sana_data/` (`datasets/`, `experiments/<folder>/`).
- **One ad-hoc run on your own images:** `s = SanaRunner(source="folder", dataset_dir="/content/my_set",
  preview_prompts=["..."])`, then `s.setup(); s.prepare_dataset(); s.build_configs(); s.train();
  s.evaluate(["a prompt", ...]); s.backup()` (the backup goes to a new `experiments/adhoc_…/` folder of the repo).
  Captions are natural language (Sana reads sentences).
- **Other checkpoints:** `variant="600m-1024"`, `"1600m-512"`, `"1600m-1024"` or `"1600m-1024-bf16"`; the training
  resolution follows the checkpoint.
- **Using a LoRA:** on a diffusers `SanaPipeline`,
  `pipe.load_lora_weights("experiments/<folder>/lora/epochN", weight_name="adapter_model.safetensors")`.
- **Updating logic = `git pull`** (the bootstrap cell pulls every run); nothing here needs editing.